<a href="https://colab.research.google.com/github/Orliluq/Inmersion_Agentes_de_IA_Alura_Clase_2/blob/main/Inmersi%C3%B3n_Agentes_de_IA_Alura_Clase_2_Orli.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
from google import genai

cliente = genai.Client()

In [ ]:
respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents="Cuál es la capital y la ciudad más grande de Turquía?"
)

print(respuesta.text)

La capital de Turquía es **Ankara**.

La ciudad más grande y poblada de Turquía es **Estambul**.

Es importante destacar que, aunque Estambul es la ciudad más conocida, grande y un importante centro cultural y económico, Ankara es la capital política y administrativa del país desde 1923.


## 📄 Paso 1: Carga de documentos

En esta etapa se cargan archivos PDF y se convierten en objetos `Document`.
Cada página se trata como una unidad independiente de información.

In [ ]:
from google.colab import files

os.makedirs("PDFs", exist_ok=True)
uploader = files.upload()

for archivo in uploader.keys():
  os.rename(archivo, f"PDFs/{archivo}")

Saving Carrarurquia_Reporte_Q1_2025.pdf to Carrarurquia_Reporte_Q1_2025.pdf
Saving Carrarurquia_Reporte_Q2_2025.pdf to Carrarurquia_Reporte_Q2_2025.pdf
Saving Carrarurquia_Reporte_Q3_2025.pdf to Carrarurquia_Reporte_Q3_2025.pdf
Saving Carrarurquia_Reporte_Q4_2025.pdf to Carrarurquia_Reporte_Q4_2025.pdf


In [ ]:
!pip install -q langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
documentos = []

for archivo2 in os.listdir("PDFs"):
  ruta = os.path.join("PDFs", archivo2)
  loader = PyPDFLoader(ruta)
  paginas = loader.load()
  documentos.extend(paginas)

In [ ]:
documentos[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q2 2025', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q2_2025.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Carrarurquía\nReporte trimestral Q2 2025 (ficticio)\nPeriodo: 01/04/2025 - 30/06/2025\nMoneda: USD\n1\n Carrarurquía\n Reporte trimestral Q2 2025 · Viajes a Turquía (ficticio)\n Periodo: 01/04/2025 - 30/06/2025\n Moneda de referencia: dólares estadounidenses (USD)\nEste documento presenta resultados y aprendizajes ficticios del trimestre para Carrarurquía, una\nempresa de viajes especializada en experiencias en Turquía. Los datos, cifras y testimonios han sido\ncreados con fines demostrativos y no representan operaciones reales.\n Edición: Julio de 2025')

In [ ]:
len(documentos)

60

In [ ]:
!pip install -q langchain-text-splitters

## ✂️ Paso 2: División en fragmentos

Los documentos se dividen en fragmentos pequeños para:

- mejorar la calidad del embedding
- evitar límites de tokens
- facilitar la recuperación semántica

Se usa `RecursiveCharacterTextSplitter` con overlap para mantener contexto.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

fragmentos = divisor.split_documents(documentos)

In [ ]:
fragmentos[67]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q4 2025', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q4_2025.pdf', 'total_pages': 15, 'page': 10, 'page_label': '11'}, page_content='Riesgo\nProbabilidad\nImpacto\nMitigación principal\nSobrecupo hotelero en picos\nMedia\nAlto\nCupos firmes + lista de hoteles\nalternativos con precios\npre-negociados.\nIncidencias en traslados\ninternos\nMedia\nMedio\nMonitoreo de rutas + buffer de horarios\n+ partners con back-up.\nVariación de precios de\nproveedores\nAlta\nMedio\nCláusulas de ajuste y revisión\nquincenal de tarifas en temporada.')

## 🧠 Paso 3: Generación de embeddings

Cada fragmento se transforma en un vector numérico usando Gemini.

Esto permite comparar significado en lugar de texto literal.

In [ ]:
!pip install -q langchain-google-genai faiss-cpu tenacity

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    batch_size=20 # Reduced batch size to manage API quota
)

## ⚡ Paso 4: Vector Store (FAISS)

Los embeddings se almacenan en FAISS para permitir búsquedas rápidas por similitud.

Se implementan reintentos con `tenacity` para manejar errores de cuota.

In [ ]:
len(fragmentos)

import logging
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type, after_log
from langchain_google_genai._common import GoogleGenerativeAIError

# Set up a basic logger for tenacity
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

@retry(
    wait=wait_exponential(multiplier=1, min=10, max=600),  # Wait 10s to 10min between retries
    stop=stop_after_attempt(10),  # Try up to 10 times
    retry=retry_if_exception_type(GoogleGenerativeAIError),
    after=after_log(logger, logging.INFO)
)
def create_faiss_vectorstore_robustly(documents, embedding_model):
    print("Intentando crear FAISS con reintentos...")
    return FAISS.from_documents(
        documents=documents,
        embedding=embedding_model
    )

In [ ]:
try:
    vectorstore1 = create_faiss_vectorstore_robustly(
        documents=fragmentos[0:89],
        embedding_model=embeddings
    )
    print("vectorstore1 creado exitosamente.")
except GoogleGenerativeAIError as e:
    print(f"💥 ERROR: No se pudo crear vectorstore1 después de múltiples reintentos. {e}")
except Exception as e:
    print(f"💥 ERROR INESPERADO al crear vectorstore1: {type(e)} {e}")

Intentando crear FAISS con reintentos...
vectorstore1 creado exitosamente.


In [ ]:
try:
    vectorstore2 = create_faiss_vectorstore_robustly(
        documents=fragmentos[90:],
        embedding_model=embeddings
    )
    print("vectorstore2 creado exitosamente.")
except GoogleGenerativeAIError as e:
    print(f"💥 ERROR: No se pudo crear vectorstore2 después de múltiples reintentos. {e}")
except Exception as e:
    print(f"💥 ERROR INESPERADO al crear vectorstore2: {type(e)} {e}")

Intentando crear FAISS con reintentos...
Intentando crear FAISS con reintentos...
vectorstore2 creado exitosamente.


In [ ]:
vectorstore1.index.reconstruct(0)

array([-0.00143831,  0.00889298,  0.01531782, ...,  0.01553657,
       -0.00974552, -0.02190461], dtype=float32)

In [ ]:
len(vectorstore1.index.reconstruct(0))

3072

In [ ]:
consulta = "Cuál es el paquete de viajes más económico de Carrarurquía?"

resultados = vectorstore1.similarity_search(
    consulta,
    k=7
)

for i in resultados:
  print(i)
  print("\n")

page_content='Carrarurquía
Reporte trimestral Q4 2025 (ficticio)
Periodo: 01/10/2025 - 31/12/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populares. Los importes se muestran por persona en ocupación' metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q4 2025', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q4_2025.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}


page_content='Carrarurquía
Reporte trimestral Q2 2025 (ficticio)
Periodo: 01/04/2025 - 30/06/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populares. Los importes se muestran

## 🔍 Paso 5: Recuperación de contexto

Se configura un retriever que obtiene los fragmentos más relevantes
según la consulta del usuario.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

retriever = vectorstore1.as_retriever(
    search_kwargs={"k": 4}
)

## 🤖 Paso 6: Generación con RAG

Se construye un prompt que incluye:

- contexto recuperado
- instrucción estricta

El modelo responde SOLO con base en ese contexto.

In [ ]:
def preguntar_rag(pregunta):
    """Busca contexto relevante en los documentos y genera una respuesta."""
    # Paso 1: Buscar los chunks más relevantes
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Paso 2: Construir el prompt con el contexto encontrado
    prompt = f"""Eres un asistente experto que responde preguntas basándose ÚNICAMENTE
    en el contexto proporcionado. Si la información no está en el contexto,
    di que no tienes suficiente información.

    Contexto: {contexto}

    Pregunta: {pregunta}

    Respuesta:"""

    # Paso 3: Enviar al modelo y devolver la respuesta
    respuesta = llm.invoke(prompt)
    return respuesta.content

In [ ]:
preguntar_rag("Dónde se mantuvo concentrado el mix de productos?")

'El mix de productos se mantuvo concentrado en circuitos combinados (Estambul + Capadocia).'

In [ ]:
preguntar_rag("Cuántos mundiales de fútbol tiene Brasil?")

'No tengo suficiente información en el contexto proporcionado para responder cuántos mundiales de fútbol tiene Brasil.'

In [ ]:
print(f"Vectorstore created with {len(vectorstore1.index.reconstruct(0))} dimensions and {vectorstore1.index.ntotal} vectors.")

Vectorstore created with 3072 dimensions and 89 vectors.


In [ ]:
print(f"Número total de vectores: {vectorstore1.index.ntotal}")
print(f"Dimensiones de los vectores: {len(vectorstore1.index.reconstruct(0))}")

Número total de vectores: 89
Dimensiones de los vectores: 3072


## ⚠️ Limitaciones

- El sistema depende de la calidad del chunking
- Puede fallar por límites de API (429)
- No razona fuera del contexto recuperado
- Es recomendable trabajar con subconjuntos o usar embeddings locales

## Explicación Paso a Paso del Código

Este notebook implementa un sistema de Recuperación Aumentada por Generación (RAG) utilizando documentos PDF, la biblioteca LangChain para procesamiento de texto y la API de Gemini para incrustaciones (embeddings) y generación de texto. A continuación, se detalla cada paso:

### 1. Instalación de Librerías y Configuración Inicial

- **`!pip install -q google-genai`**: Instala la librería de Google Generative AI, necesaria para interactuar con los modelos de Gemini.
- **`!pip install -q langchain-community pypdf`**: Instala `langchain-community` (para componentes de LangChain) y `pypdf` (para cargar documentos PDF).
- **`!pip install -q langchain-text-splitters`**: Instala `langchain-text-splitters`, utilizada para dividir textos grandes en fragmentos más pequeños.
- **`!pip install -q langchain-google-genai faiss-cpu tenacity`**: Instala `langchain-google-genai` (integración de Gemini con LangChain), `faiss-cpu` (para el almacén vectorial FAISS) y `tenacity` (para añadir reintentos robustos a las llamadas a la API).
- **`from google.colab import userdata` y `os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')`**: Configura la clave de API de Gemini, obteniéndola de los secretos de Colab por seguridad y asignándola a una variable de entorno.
- **`from google import genai` y `cliente = genai.Client()`**: Inicializa el cliente de la API de Gemini.

### 2. Carga y Preparación de Documentos PDF

- **`os.makedirs("PDFs", exist_ok=True)` y `uploader = files.upload()`**: Crea una carpeta llamada `PDFs` y permite al usuario subir archivos PDF. Los PDFs cargados se guardan en esta carpeta.
- **`for archivo in uploader.keys(): os.rename(archivo, f"PDFs/{archivo}")`**: Renombra los archivos subidos para que estén en la carpeta `PDFs`.
- **`from langchain_community.document.loaders import PyPDFLoader`**: Importa el cargador de PDF de LangChain.
- **`documentos = []` y bucle para cargar PDFs**: Itera sobre los archivos en la carpeta `PDFs`. Para cada archivo, `PyPDFLoader` lo carga y extrae sus páginas. Estas páginas se añaden a la lista `documentos`.
- **`documentos[0]` y `len(documentos)`**: Muestran un ejemplo del primer documento cargado y el número total de páginas (documentos) procesados.

### 3. División de Documentos en Fragmentos (Chunking)

- **`from langchain_text_splitters import RecursiveCharacterTextSplitter`**: Importa la clase para dividir texto de forma recursiva.
- **`divisor = RecursiveCharacterTextSplitter(...)`**: Define un `RecursiveCharacterTextSplitter` con un tamaño de fragmento (`chunk_size`) de 400 caracteres, una superposición (`chunk_overlap`) de 40 caracteres y una lista de separadores. Esto asegura que los fragmentos no sean demasiado grandes y que haya suficiente contexto entre ellos.
- **`fragmentos = divisor.split_documents(documentos)`**: Divide todos los documentos cargados en fragmentos más pequeños utilizando el `divisor` configurado.
- **`fragmentos[67]`**: Muestra un ejemplo de uno de los fragmentos resultantes.

### 4. Generación de Incrustaciones (Embeddings) y Almacenamiento Vectorial

- **`from langchain_google_genai import GoogleGenerativeAIEmbeddings`**: Importa la clase para generar incrustaciones usando modelos de Gemini.
- **`from langchain_community.vectorstores import FAISS`**: Importa FAISS, un almacén vectorial eficiente para búsquedas de similitud.
- **`embeddings = GoogleGenerativeAIEmbeddings(...)`**: Inicializa el modelo de incrustaciones de Gemini (`gemini-embedding-001`) con un `batch_size` reducido para gestionar las cuotas de la API.
- **Función `create_faiss_vectorstore_robustly` con `tenacity`**: Esta función envuelve la creación del almacén FAISS y utiliza la librería `tenacity` para implementar reintentos con espera exponencial. Esto es crucial para manejar errores temporales de la API (como los `RESOURCE_EXHAUSTED` debido a límites de cuota).
- **Creación de `vectorstore1` y `vectorstore2`**: Los fragmentos se dividen en dos grupos (`fragmentos[0:89]` y `fragmentos[90:]`) para crear dos almacenes vectoriales FAISS separados, probablemente para manejar grandes volúmenes de datos o para demostrar la escalabilidad. La función `create_faiss_vectorstore_robustly` se invoca para cada uno.
- **`vectorstore1.index.reconstruct(0)` y `len(vectorstore1.index.reconstruct(0))`**: Muestran el primer vector de incrustación almacenado en `vectorstore1` y su dimensionalidad, confirmando que las incrustaciones se han generado correctamente.

### 5. Configuración del Modelo de Lenguaje (LLM) y Recuperador (Retriever)

- **`from langchain_google_genai import ChatGoogleGenerativeAI`**: Importa la clase para usar los modelos de chat de Gemini.
- **`llm = ChatGoogleGenerativeAI(...)`**: Inicializa el modelo de chat (`gemini-2.5-flash`) con una `temperature` de 0.2, lo que lo hace más determinista en sus respuestas.
- **`retriever = vectorstore1.as_retriever(search_kwargs={"k": 4})`**: Configura un recuperador a partir de `vectorstore1`. Este recuperador buscará los 4 fragmentos (`k=4`) más relevantes cuando se le haga una consulta.

### 6. Implementación del Sistema RAG (Preguntas y Respuestas)

- **Función `preguntar_rag(pregunta)`**: Esta función es el corazón del sistema RAG:
    1. **`docs = retriever.invoke(pregunta)`**: Toma una `pregunta` del usuario y usa el `retriever` para encontrar los fragmentos de documento más relevantes en `vectorstore1`.
    2. **`contexto = "\n\n---\n\n".join(doc.page_content for doc in docs)`**: Concatena el contenido de los documentos recuperados para formar un `contexto` que se le pasará al LLM.
    3. **`prompt = f"Eres un asistente experto..."`**: Construye un prompt para el LLM. Este prompt instruye al modelo a responder `ÚNICAMENTE` basándose en el `contexto` proporcionado y a indicar si no tiene suficiente información.
    4. **`respuesta = llm.invoke(prompt)`**: Envía el prompt y el contexto al modelo de chat de Gemini y obtiene una `respuesta`.
    5. **`return respuesta.content`**: Devuelve la respuesta generada por el LLM.

### 7. Ejemplos de Uso del RAG

- **`preguntar_rag("Dónde se mantuvo concentrado el mix de productos?")`**: Demuestra una consulta donde la respuesta se encuentra en los documentos cargados.
- **`preguntar_rag("Cuántos mundiales de fútbol tiene Brasil?")`**: Demuestra una consulta donde la información no está en los documentos, y el sistema responde que no tiene suficiente información, validando la directriz del prompt.

En resumen, el código ingiere y procesa documentos PDF, crea incrustaciones para ellos, los almacena en una base de datos vectorial para una búsqueda eficiente, y luego utiliza un modelo de lenguaje grande (LLM) para generar respuestas informadas por el contexto recuperado de esos documentos.